# Parking Redevelopment Opportunity Analysis

**GIS / Remote Spatial Analysis / Multi-Criteria Decision Analysis**

This portfolio project identifies and ranks parking-lot redevelopment candidates within a defined study area in **Dallas, Texas**.

The analysis integrates:
- Overture Maps parking, building and land-use data
- DART GTFS transit-stop data
- GeoPandas spatial joins, overlays, buffers and nearest-feature analysis
- A transparent **Multi-Criteria Score (MCS)**
- GIS-ready GeoPackage and CSV outputs
- An interactive Folium map

> **Portfolio interpretation:** The score is a project-defined analytical framework for identifying sites for *further investigation*. It is not an official planning recommendation.


## 1. Analytical question

**Which parking lots have spatial characteristics that make them interesting candidates for further redevelopment analysis?**

Five criteria are combined:

1. **Transit accessibility** — proximity to a DART stop
2. **Lot size** — relative parking-lot area
3. **Green-space context** — green/open-space coverage within 500 m
4. **Development context** — building coverage within 500 m
5. **Parking concentration** — area of other parking within 500 m

Each criterion is converted to a 0–100 score and combined using documented weights.


## 2. Reproducible project structure

This notebook contains **no machine-specific paths** such as `C:\Users\...`.

Recommended GitHub structure:

```text
parking-redevelopment-analysis/
├── notebooks/
│   └── parking_redevelopment_analysis_portfolio.ipynb
├── data/
│   └── dart_gtfs.zip
├── outputs/
│   ├── parking_redevelopment_final.gpkg
│   ├── parking_redevelopment_ranked_results.csv
│   └── parking_redevelopment_final_map.html
└── README.md
```

The notebook uses three portable path variables:
- `PROJECT_DIR`
- `DATA_DIR`
- `OUTPUT_DIR`


In [ ]:
from pathlib import Path
import zipfile
import requests

import numpy as np
import pandas as pd
import geopandas as gpd
import folium

from shapely.geometry import box
from overturemaps import geodataframe
from IPython.display import IFrame, display

# Repository-friendly paths.
# Run the notebook from the repository's notebooks/ directory.
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_CRS = "EPSG:4326"
WORKING_CRS = "EPSG:32614"   # WGS 84 / UTM Zone 14N
BUFFER_M = 500

print("Project:", PROJECT_DIR)
print("Data:   ", DATA_DIR)
print("Outputs:", OUTPUT_DIR)


## 3. Study area

The original analysis uses the following geographic bounding box:

- West: `-96.8166593`
- South: `32.7689828`
- East: `-96.7790675`
- North: `32.7961922`

All area and distance calculations are performed in **EPSG:32614**, so measurements are in metres rather than geographic degrees.


In [ ]:
BBOX = (
    -96.8166593,
    32.7689828,
    -96.7790675,
    32.7961922
)

study_area = gpd.GeoDataFrame(
    {"geometry": [box(*BBOX)]},
    crs=SOURCE_CRS
)
study_area_projected = study_area.to_crs(WORKING_CRS)

print("Study-area CRS:", study_area.crs)
print("Working CRS:", study_area_projected.crs)
print(
    f"Study area: "
    f"{study_area_projected.geometry.area.iloc[0] / 1_000_000:.2f} km²"
)


## 4. Parking inventory

Parking polygons are extracted from the Overture Maps `infrastructure` layer.

Only features classified as `parking` and represented as polygons or multipolygons are retained. These become the candidate population for the scoring model.


In [ ]:
infrastructure = geodataframe("infrastructure", bbox=BBOX)

parking = infrastructure[
    infrastructure["class"].astype(str).str.lower().eq("parking")
].copy()

parking_polygons = parking[
    parking.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()

if parking_polygons.crs is None:
    parking_polygons = parking_polygons.set_crs(SOURCE_CRS)

parking_projected = parking_polygons.to_crs(WORKING_CRS)

print(f"Parking polygon features: {len(parking_projected):,}")
print(parking_projected.geometry.geom_type.value_counts())


## 5. Neighbourhood context

A **500 m radius** is used consistently around each parking lot for neighbourhood-level context.

The same spatial window is used for:
- green-space coverage
- building coverage
- nearby parking concentration

Using a common scale makes these contextual indicators comparable.


In [ ]:
parking_buffer = parking_projected[["id", "geometry"]].copy()
parking_buffer["geometry"] = parking_buffer.geometry.buffer(BUFFER_M)

parking_buffer_dissolved = parking_buffer.dissolve()

print("Parking lots:", len(parking_buffer))
print("Buffer radius:", BUFFER_M, "m")


## 6. Transit accessibility

DART GTFS stop data are downloaded into `DATA_DIR` if they are not already present.

For each parking lot, the nearest DART stop is identified using a spatial nearest-neighbour join.

### Transit score

Distance is converted to a relative 0–100 score:

\[
TransitScore_i =
100 \times
\frac{d_{max}-d_i}{d_{max}-d_{min}}
\]

A shorter distance therefore produces a higher score.

This is a **relative score within the study area**, not a universal accessibility standard.


In [ ]:
GTFS_URL = "https://www.dart.org/transitdata/latest/google_transit.zip"
GTFS_FILE = DATA_DIR / "dart_gtfs.zip"

if not GTFS_FILE.exists():
    response = requests.get(GTFS_URL, timeout=60)
    response.raise_for_status()
    GTFS_FILE.write_bytes(response.content)
    print("Downloaded GTFS:", GTFS_FILE)
else:
    print("Using existing GTFS:", GTFS_FILE)

with zipfile.ZipFile(GTFS_FILE, "r") as z:
    stops = pd.read_csv(z.open("stops.txt"))

transit_stops = gpd.GeoDataFrame(
    stops,
    geometry=gpd.points_from_xy(
        stops.stop_lon,
        stops.stop_lat
    ),
    crs=SOURCE_CRS
).to_crs(WORKING_CRS)

transit_stops_study = gpd.sjoin(
    transit_stops,
    study_area_projected,
    predicate="within",
    how="inner"
)

parking_transit = gpd.sjoin_nearest(
    parking_projected,
    transit_stops_study[["stop_id", "stop_name", "geometry"]],
    how="left",
    distance_col="distance_to_transit_m"
)

min_dist = parking_transit["distance_to_transit_m"].min()
max_dist = parking_transit["distance_to_transit_m"].max()

parking_transit["transit_score"] = (
    100
    * (max_dist - parking_transit["distance_to_transit_m"])
    / (max_dist - min_dist)
)

print("DART stops in study area:", len(transit_stops_study))
print(parking_transit[[
    "id", "stop_name", "distance_to_transit_m", "transit_score"
]].head())


## 7. Parking-lot size score

Parking area is calculated from the projected polygon geometry.

The original scoring approach uses **quintiles**:

| Relative lot size | Score |
|---|---:|
| Bottom 20% | 20 |
| 20–40% | 40 |
| 40–60% | 60 |
| 60–80% | 80 |
| Top 20% | 100 |

This makes lot size a relative measure within the study population.


In [ ]:
parking_transit["parking_area_m2"] = parking_transit.geometry.area
area = parking_transit["parking_area_m2"]

parking_transit["size_score"] = pd.cut(
    area,
    bins=[
        -np.inf,
        area.quantile(0.20),
        area.quantile(0.40),
        area.quantile(0.60),
        area.quantile(0.80),
        np.inf
    ],
    labels=[20, 40, 60, 80, 100],
    include_lowest=True
).astype(int)

print(parking_transit["parking_area_m2"].describe())
print(parking_transit["size_score"].value_counts().sort_index())


## 8. Green-space context

Green/open-space features are extracted from Overture land-use data using the classes retained in the original analysis:

`grass`, `park`, `recreation_ground`, `garden`, `flowerbed`, `dog_park`, `greenfield`, `protected`, and `cemetery`.

For each candidate, green-space polygons intersecting the 500 m buffer are overlaid and their areas are summed.

### Green score

Green coverage is divided into quintiles. The original model assigns **higher scores to higher green coverage**, treating surrounding green space as a potential redevelopment-design opportunity.

This is a project-defined interpretation, not an official planning standard.


In [ ]:
GREEN_CLASSES = [
    "grass",
    "park",
    "recreation_ground",
    "garden",
    "flowerbed",
    "dog_park",
    "greenfield",
    "protected",
    "cemetery"
]

land_use = geodataframe("land_use", bbox=BBOX)

green_space = land_use[
    land_use["class"].isin(GREEN_CLASSES)
].copy()

if green_space.crs is None:
    green_space = green_space.set_crs(SOURCE_CRS)

green_space_projected = green_space.to_crs(WORKING_CRS)

parking_green_buffers = parking_transit[["id", "geometry"]].copy()
parking_green_buffers["geometry"] = (
    parking_green_buffers.geometry.buffer(BUFFER_M)
)

green_intersection = gpd.overlay(
    parking_green_buffers,
    green_space_projected[["class", "geometry"]],
    how="intersection"
)

green_intersection["green_area_m2"] = (
    green_intersection.geometry.area
)

green_area_by_parking = (
    green_intersection
    .groupby("id")["green_area_m2"]
    .sum()
    .reset_index()
)

parking_green = parking_transit.merge(
    green_area_by_parking,
    on="id",
    how="left"
)

parking_green["green_area_m2"] = (
    parking_green["green_area_m2"].fillna(0)
)

buffer_area_m2 = np.pi * BUFFER_M**2

parking_green["green_coverage_pct"] = (
    parking_green["green_area_m2"] / buffer_area_m2
) * 100

parking_green["green_score"] = pd.qcut(
    parking_green["green_coverage_pct"],
    q=5,
    labels=[100, 80, 60, 40, 20],
    duplicates="drop"
).astype(int)

print("Green-space features:", len(green_space_projected))
print(parking_green["green_score"].value_counts().sort_index())


## 9. Building-context score

Building footprints are downloaded from Overture Maps using four non-overlapping tiles covering the study area.

For each parking lot, building footprints intersecting its 500 m buffer are summed.

### Development score

Higher building coverage receives a higher relative score:

| Building-coverage quintile | Score |
|---|---:|
| Bottom 20% | 20 |
| 20–40% | 40 |
| 40–60% | 60 |
| 60–80% | 80 |
| Top 20% | 100 |

This is a **development-context proxy** rather than a claim that higher building coverage is inherently better.


In [ ]:
west, south, east, north = BBOX
mid_lon = (west + east) / 2
mid_lat = (south + north) / 2

tiles = [
    (west, south, mid_lon, mid_lat),
    (mid_lon, south, east, mid_lat),
    (west, mid_lat, mid_lon, north),
    (mid_lon, mid_lat, east, north)
]

building_tiles = [
    geodataframe("building", bbox=tile)
    for tile in tiles
]

buildings = gpd.GeoDataFrame(
    pd.concat(building_tiles, ignore_index=True),
    crs=SOURCE_CRS
).to_crs(WORKING_CRS)

print("Building features:", len(buildings))


In [ ]:
parking_building = parking_green[["id", "geometry"]].copy()
parking_building["geometry"] = (
    parking_building.geometry.buffer(BUFFER_M)
)

building_intersection = gpd.overlay(
    parking_building,
    buildings[["geometry"]],
    how="intersection"
)

building_intersection["building_area_m2"] = (
    building_intersection.geometry.area
)

building_area_by_parking = (
    building_intersection
    .groupby("id")["building_area_m2"]
    .sum()
    .reset_index()
)

parking_green = parking_green.merge(
    building_area_by_parking,
    on="id",
    how="left"
)

parking_green["building_area_m2"] = (
    parking_green["building_area_m2"].fillna(0)
)

parking_green["building_coverage_pct"] = (
    parking_green["building_area_m2"] / buffer_area_m2
) * 100

parking_green["development_score"] = pd.qcut(
    parking_green["building_coverage_pct"],
    q=5,
    labels=[20, 40, 60, 80, 100],
    duplicates="drop"
).astype(int)

print(parking_green["development_score"].value_counts().sort_index())


## 10. Parking concentration score

Parking concentration measures the area of **other parking lots within 500 m** of each candidate.

The candidate itself is excluded from its neighbourhood total.

Higher nearby parking concentration receives a higher quintile score.

This criterion is intended to highlight locations where parking supply is spatially concentrated and may warrant further land-use investigation.


In [ ]:
parking_concentration = parking_green[["id", "geometry"]].copy()
parking_concentration["buffer_500m"] = (
    parking_concentration.geometry.buffer(BUFFER_M)
)

nearby = gpd.sjoin(
    parking_concentration[["id", "buffer_500m"]].set_geometry("buffer_500m"),
    parking_concentration[["id", "geometry"]],
    how="left",
    predicate="intersects",
    lsuffix="_candidate",
    rsuffix="_nearby"
)

area_by_id = parking_concentration.set_index("id").geometry.area

nearby["nearby_parking_area_m2"] = (
    nearby["id__nearby"].map(area_by_id)
)

nearby_other = nearby[
    nearby["id__candidate"] != nearby["id__nearby"]
].copy()

nearby_area = (
    nearby_other
    .groupby("id__candidate")["nearby_parking_area_m2"]
    .sum()
    .reset_index()
    .rename(columns={"id__candidate": "id"})
)

parking_green = parking_green.merge(
    nearby_area,
    on="id",
    how="left"
)

parking_green["nearby_parking_area_m2"] = (
    parking_green["nearby_parking_area_m2"].fillna(0)
)

parking_green["concentration_score"] = pd.qcut(
    parking_green["nearby_parking_area_m2"],
    q=5,
    labels=[20, 40, 60, 80, 100],
    duplicates="drop"
).astype(int)

print(parking_green["concentration_score"].value_counts().sort_index())


## 11. Multi-Criteria Score (MCS)

The five normalized criteria are combined into a weighted score:

\[
MCS =
0.30T +
0.25S +
0.20G +
0.15D +
0.10C
\]

Where:

- **T** = transit accessibility
- **S** = parking-lot size
- **G** = green-space context
- **D** = development context
- **C** = parking concentration

| Criterion | Weight |
|---|---:|
| Transit accessibility | 30% |
| Lot size | 25% |
| Green-space context | 20% |
| Development context | 15% |
| Parking concentration | 10% |
| **Total** | **100%** |

The weights are explicit project assumptions so that the model can be audited or modified.


In [ ]:
score_columns = [
    "transit_score",
    "size_score",
    "green_score",
    "development_score",
    "concentration_score"
]

parking_green["final_score"] = (
    parking_green["transit_score"] * 0.30
    + parking_green["size_score"] * 0.25
    + parking_green["green_score"] * 0.20
    + parking_green["development_score"] * 0.15
    + parking_green["concentration_score"] * 0.10
)

parking_green["rank"] = (
    parking_green["final_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

parking_green["priority"] = np.select(
    [
        parking_green["final_score"] >= 80,
        parking_green["final_score"] >= 50
    ],
    [
        "High Priority",
        "Moderate Priority"
    ],
    default="Low Priority"
)

print("Final score:")
print(parking_green["final_score"].describe())

print("\nPriority classes:")
print(parking_green["priority"].value_counts())


## 12. QA / validation

The portfolio version keeps the substantive QA checks from the original workflow and removes exploratory/debugging cells.

The checks cover:

- projected CRS
- empty geometries
- invalid geometries
- stable feature count
- 0–100 criterion ranges
- 0–100 final-score range
- unique candidate IDs
- unique ranking values

Keeping these checks visible demonstrates that the workflow is designed for **reproducibility and data quality**, not just visualization.


In [ ]:
print("CRS:", parking_green.crs)
print("Empty geometries:", parking_green.geometry.is_empty.sum())
print("Invalid geometries:", (~parking_green.geometry.is_valid).sum())
print("Parking lots scored:", len(parking_green))

for column in score_columns + ["final_score"]:
    print(
        f"{column:24s} "
        f"min={parking_green[column].min():.2f} "
        f"max={parking_green[column].max():.2f}"
    )

assert parking_green.geometry.is_empty.sum() == 0
assert parking_green.geometry.is_valid.all()
assert parking_green["final_score"].between(0, 100).all()

print("QA checks completed successfully.")


## 13. Ranked candidate output

The ranked table exposes both the final score and the component metrics, allowing a reviewer to understand **why** a candidate received its position instead of treating the MCS as a black box.


In [ ]:
parking_green = parking_green.sort_values(
    "final_score",
    ascending=False
).copy()

parking_green["candidate_id"] = (
    "DAL-PARK-"
    + parking_green["rank"].astype(int).astype(str).str.zfill(3)
)

final_results = parking_green[
    [
        "candidate_id",
        "rank",
        "final_score",
        "priority",
        "parking_area_m2",
        "stop_name",
        "distance_to_transit_m",
        "transit_score",
        "green_coverage_pct",
        "green_score",
        "building_coverage_pct",
        "development_score",
        "nearby_parking_area_m2",
        "concentration_score"
    ]
].sort_values("rank")

final_results.head(20)


## 14. Interactive map

The final Folium map includes:

- parking-lot polygons
- MCS priority category
- rank and final score
- nearest DART stop and distance
- green and building coverage
- all five criterion scores
- DART stops
- study-area boundary

The map is exported as a standalone HTML file suitable for a GitHub repository or portfolio website.


In [ ]:
def priority_color(score):
    if score >= 80:
        return "green"
    elif score >= 50:
        return "orange"
    return "red"

parking_map = parking_green.to_crs(SOURCE_CRS).copy()
parking_map["map_color"] = parking_map["final_score"].apply(priority_color)

dart_stops_map = transit_stops_study.to_crs(SOURCE_CRS).copy()
study_wgs84 = study_area_projected.to_crs(SOURCE_CRS)

center = study_wgs84.geometry.iloc[0].centroid

m_final = folium.Map(
    location=[center.y, center.x],
    zoom_start=14,
    tiles="CartoDB positron"
)

parking_layer = folium.FeatureGroup(
    name="Parking Lots — MCS Priority",
    show=True
)

for _, row in parking_map.iterrows():
    color = row["map_color"]

    popup_html = (
        f"<b>{row['candidate_id']}</b><br>"
        f"Rank: #{int(row['rank'])}<br>"
        f"MCS: {row['final_score']:.2f}<br>"
        f"Priority: {row['priority']}<hr>"
        f"Parking area: {row['parking_area_m2']:,.0f} m²<br>"
        f"Nearest DART stop: {row['stop_name']}<br>"
        f"Transit distance: {row['distance_to_transit_m']:.1f} m<hr>"
        f"Green coverage: {row['green_coverage_pct']:.1f}%<br>"
        f"Building coverage: {row['building_coverage_pct']:.1f}%<br>"
        f"Nearby parking: {row['nearby_parking_area_m2']:,.0f} m²<hr>"
        f"<b>Criterion scores</b><br>"
        f"Transit: {row['transit_score']:.1f}<br>"
        f"Lot size: {int(row['size_score'])}<br>"
        f"Green opportunity: {int(row['green_score'])}<br>"
        f"Development: {int(row['development_score'])}<br>"
        f"Parking concentration: {int(row['concentration_score'])}"
    )

    folium.GeoJson(
        row["geometry"],
        style_function=lambda feature, color=color: {
            "fillColor": color,
            "color": color,
            "weight": 1,
            "fillOpacity": 0.55
        },
        highlight_function=lambda feature: {
            "weight": 3,
            "fillOpacity": 0.8
        },
        tooltip=(
            f"{row['candidate_id']} | "
            f"Rank #{int(row['rank'])} | "
            f"MCS {row['final_score']:.2f}"
        ),
        popup=folium.Popup(popup_html, max_width=350)
    ).add_to(parking_layer)

parking_layer.add_to(m_final)

dart_layer = folium.FeatureGroup(
    name="DART Transit Stops",
    show=True
)

for _, row in dart_stops_map.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        fill=True,
        fill_opacity=0.9,
        tooltip=row["stop_name"],
        popup=folium.Popup(
            f"<b>DART Stop</b><br>{row['stop_name']}<br>"
            f"Stop ID: {row['stop_id']}",
            max_width=300
        )
    ).add_to(dart_layer)

dart_layer.add_to(m_final)

folium.GeoJson(
    study_wgs84.geometry.iloc[0],
    style_function=lambda feature: {
        "color": "black",
        "weight": 3,
        "fillOpacity": 0
    },
    tooltip="Study Area"
).add_to(m_final)

folium.LayerControl(collapsed=False).add_to(m_final)

map_path = OUTPUT_DIR / "parking_redevelopment_final_map.html"
m_final.save(map_path)

print("Map saved:", map_path)
display(IFrame(src=str(map_path), width="100%", height=750))


## 15. Export final GIS deliverables

The final outputs are written to `OUTPUT_DIR`:

- `parking_redevelopment_final.gpkg` — spatial analysis results and DART stops
- `parking_redevelopment_ranked_results.csv` — ranked candidate table
- `parking_redevelopment_final_map.html` — interactive map


In [ ]:
gpkg_path = OUTPUT_DIR / "parking_redevelopment_final.gpkg"
csv_path = OUTPUT_DIR / "parking_redevelopment_ranked_results.csv"

parking_green.to_file(
    gpkg_path,
    layer="parking_candidates",
    driver="GPKG"
)

transit_stops_study.to_file(
    gpkg_path,
    layer="dart_stops",
    driver="GPKG"
)

final_results.to_csv(csv_path, index=False)

print("GeoPackage:", gpkg_path)
print("CSV:", csv_path)
print("HTML:", map_path)


## 16. Portfolio summary

### What this project demonstrates

**Data acquisition → spatial preprocessing → proximity analysis → spatial overlay → scoring → QA → GIS export → interactive visualization**

Technical skills demonstrated:

- Python geospatial analysis
- GeoPandas / Shapely
- Overture Maps data
- GTFS ingestion
- CRS and projected-distance management
- Spatial joins and nearest-neighbour analysis
- Buffer and overlay operations
- Relative classification / normalization
- Multi-Criteria Decision Analysis (MCDA)
- Reproducible project paths
- GIS QA/QC
- GeoPackage and CSV production
- Folium web mapping

### Limitations and next steps

The MCS is intentionally transparent but simplified. A production planning workflow could add:

- parcel ownership and zoning
- assessed land value
- development regulations
- pedestrian-network accessibility
- transit frequency and route connectivity
- environmental constraints
- sensitivity analysis of the weights
- stakeholder-defined scoring criteria

The strongest portfolio takeaway is the **auditable analytical workflow**, not the score itself.


## 17. Reproducibility checklist

1. Place this notebook in `notebooks/`.
2. Create/use `data/` and `outputs/` at the repository root.
3. Install the required packages: `geopandas`, `pandas`, `numpy`, `shapely`, `requests`, `folium`, and `overturemaps`.
4. Run the notebook from the `notebooks/` directory.
5. DART GTFS is downloaded automatically when missing.
6. Final GIS, CSV, and HTML outputs are written to `outputs/`.

**No personal Windows paths are required.**
